# Uncertainty forecasting on the consumer-loans dataset

This notebook extends the signature-transform forecasting pipeline from point forecasts to **probabilistic forecasts** on the original consumer-loans dataset. The uncertainty layer is implemented through **weighted quantile regression**, which produces conditional quantiles and prediction intervals while keeping the same signature features and adaptive weighting scheme.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from marketplace_signature_forecast.config import (
    ALL_HORIZONS,
    DEFAULT_DEPTH,
    DEFAULT_N_TARGET,
    DEFAULT_WINDOW_SIZE,
    END_DATE,
    FIGURES_DIR,
    FRED_API_KEY,
    START_DATE,
    N_VALIDATION,
    TARGET,
    DATA_DIR,
    ensure_directories,
)
from marketplace_signature_forecast.data_loading import build_data_dictionary, fetch_marketplace_series
from marketplace_signature_forecast.preprocessing import (
    build_model_dataset,
    prepare_standardized_arrays,
    resample_collection,
    resample_to_weekly,
    save_processed_bundle,
    train_validation_split,
)
from marketplace_signature_forecast.evaluation import run_multi_horizon_experiment
from marketplace_signature_forecast.plotting import plot_quantile_forecast_interval
from marketplace_signature_forecast.quantile_evaluation import run_multi_horizon_quantile_experiment

ensure_directories()

BASE_OUTPUT_DIR = DATA_DIR / "consumer_loans_uncertainty"
DATASET_DIR = BASE_OUTPUT_DIR / "dataset"
Y_ONLY_DIR = BASE_OUTPUT_DIR / "signature_y_only_quantiles"
JOINT_DIR = BASE_OUTPUT_DIR / "signature_joint_path_quantiles"
POINT_Y_ONLY_DIR = BASE_OUTPUT_DIR / "point_y_only"
POINT_JOINT_DIR = BASE_OUTPUT_DIR / "point_joint_path"
PLOT_DIR = FIGURES_DIR / "consumer_loans_uncertainty"

for directory in [BASE_OUTPUT_DIR, DATASET_DIR, Y_ONLY_DIR, JOINT_DIR, POINT_Y_ONLY_DIR, POINT_JOINT_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

QUANTILES = [0.1, 0.5, 0.9]
QUANTILE_ALPHA = 1e-2
WINDOW_SIZE = DEFAULT_WINDOW_SIZE
DEPTH = DEFAULT_DEPTH
N_TARGET = DEFAULT_N_TARGET
PLOT_HORIZON = 12
ADD_TIME = True

data_dictionary = build_data_dictionary()
data_dictionary

## 1. Rebuild the weekly dataset

In [ ]:
y_raw, x_raw = fetch_marketplace_series(FRED_API_KEY, START_DATE, END_DATE)
y_weekly = resample_to_weekly(y_raw)
x_weekly = resample_collection(x_raw)
full_data = build_model_dataset(y_weekly, x_weekly).dropna().copy()
train_data, validation_data = train_validation_split(full_data, N_VALIDATION)

save_processed_bundle(full_data, train_data, validation_data, data_dictionary, DATASET_DIR)

print(full_data.shape)
print(full_data.index.min(), full_data.index.max())
full_data.head()

## 2. Standardized arrays

In [ ]:
prepared = prepare_standardized_arrays(
    full_data=full_data,
    target_col=TARGET["name"],
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
)

X = prepared["X"]
y = prepared["y"]
dates = prepared["dates"]
y_scaler = prepared["y_scaler"]

print(X.shape, y.shape)

## 3. Probabilistic forecast: target-path signature only

In [ ]:
quantile_results_y_only, quantile_summary_y_only = run_multi_horizon_quantile_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    quantiles=QUANTILES,
    output_dir=Y_ONLY_DIR,
    alpha=QUANTILE_ALPHA,
    n_target=N_TARGET,
    use_sig_y_only=True,
    add_time=ADD_TIME,
)

quantile_summary_y_only = quantile_summary_y_only.rename(
    columns={
        "pinball_q0_1": "pinball_q0_1_y_only",
        "pinball_q0_5": "pinball_q0_5_y_only",
        "pinball_q0_9": "pinball_q0_9_y_only",
        "median_mae": "median_mae_y_only",
        "median_rmse": "median_rmse_y_only",
        "median_mre": "median_mre_y_only",
        "interval_coverage": "interval_coverage_y_only",
        "avg_interval_width": "avg_interval_width_y_only",
    }
)
quantile_summary_y_only

## 4. Probabilistic forecast: joint-path signature

In [ ]:
quantile_results_joint, quantile_summary_joint = run_multi_horizon_quantile_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    quantiles=QUANTILES,
    output_dir=JOINT_DIR,
    alpha=QUANTILE_ALPHA,
    n_target=N_TARGET,
    use_sig_y_only=False,
    add_time=ADD_TIME,
)

quantile_summary_joint = quantile_summary_joint.rename(
    columns={
        "pinball_q0_1": "pinball_q0_1_joint",
        "pinball_q0_5": "pinball_q0_5_joint",
        "pinball_q0_9": "pinball_q0_9_joint",
        "median_mae": "median_mae_joint",
        "median_rmse": "median_rmse_joint",
        "median_mre": "median_mre_joint",
        "interval_coverage": "interval_coverage_joint",
        "avg_interval_width": "avg_interval_width_joint",
    }
)
quantile_summary_joint

## 5. Compare probabilistic and point forecasts

In [ ]:
point_results_y_only, point_summary_y_only = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=POINT_Y_ONLY_DIR,
    use_sig_y_only=True,
    add_time=ADD_TIME,
)

point_results_joint, point_summary_joint = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=POINT_JOINT_DIR,
    use_sig_y_only=False,
    add_time=ADD_TIME,
)

summary_table = quantile_summary_y_only.merge(
    quantile_summary_joint,
    on=["horizon_weeks", "n_forecasts"],
    how="outer",
).merge(
    point_summary_y_only.rename(columns={"signature_mre": "point_mre_y_only"}),
    on=["horizon_weeks", "n_forecasts"],
    how="left",
).merge(
    point_summary_joint.rename(columns={"signature_mre": "point_mre_joint"}),
    on=["horizon_weeks", "n_forecasts"],
    how="left",
)

summary_table.to_csv(BASE_OUTPUT_DIR / "consumer_loans_uncertainty_summary.csv", index=False)
summary_table.round(3)

## 6. Prediction intervals for a 12-week horizon

In [ ]:
forecast_y_only = pd.read_csv(Y_ONLY_DIR / f"quantile_forecast_results_delta{PLOT_HORIZON}.csv")
forecast_joint = pd.read_csv(JOINT_DIR / f"quantile_forecast_results_delta{PLOT_HORIZON}.csv")

plot_quantile_forecast_interval(
    forecast_df=forecast_y_only,
    delta_t=PLOT_HORIZON,
    target_label=TARGET["name"],
    path=PLOT_DIR / f"consumer_loans_interval_y_only_delta{PLOT_HORIZON}.png",
)

plot_quantile_forecast_interval(
    forecast_df=forecast_joint,
    delta_t=PLOT_HORIZON,
    target_label=TARGET["name"],
    path=PLOT_DIR / f"consumer_loans_interval_joint_delta{PLOT_HORIZON}.png",
)

summary_table.loc[summary_table["horizon_weeks"] == PLOT_HORIZON].T